In [0]:
# configure paths
catalog = "global_mart_retail_dev"
schema = "bronze"
volume = "raw_data"
table = "superstore_orders"

# volume path
volume_path = f"/Volumes/{catalog}/{schema}/{volume}/"
print(f"Reading from: {volume_path}")

In [0]:
# import libraries 
from pyspark.sql.functions import *
from pyspark.sql.types import *
import re
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import *

In [0]:
# read csv file from volume
df = (
    spark.read
    .format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .option("delimiter",",")
    .option("quote",'"')
    .option("escape","\"")
    .option("encoding","ISO-8859-1")
    .load(volume_path)
)

# data checks 
print(f" Number of rows: {df.count()}")
print(f" Number of columns: {len(df.columns)}")
print(f" Column names: {df.columns}")
#display(df.limit(5))

In [0]:
# the columns all have spaces in between them, this is not convinent for storage in databricks
# the columns will need to be sanitised
def sanitize_column(name):
    return re.sub(r"[ ,;{}()\n\t=\-]", "_", name)

for column_name in df.columns:
    df = df.withColumnRenamed(column_name, sanitize_column(column_name))

# display dataframe
display(df.limit(5))


In [0]:
# add metadata columns for bronze layer, to help with tracing files in the pipeline
ingest_date = datetime.now().date()
df = (
    df.withColumn("ingestion_timestamp" , current_timestamp())
      .withColumn("source_file", col("_metadata.file_name"))
      .withColumn("_ingest_date",lit(ingest_date))               
)
display(df.limit(5))

In [0]:
window_spec = Window.partitionBy("Order_ID", "Product_ID").orderBy(col("ingestion_timestamp").desc())

df_bronze = (
    df
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

display(df_bronze.limit(5))
print(f"After deduplication: {df_bronze.count()} rows")

In [0]:
# writing dataframe to the bronze catalog as a delta table 
target_table = f"{catalog}.{schema}.{table}"

# merge
if spark.catalog.tableExists(target_table):
    delta_bronze = DeltaTable.forName(spark, target_table)
    delta_bronze.alias("target").merge(
        df_bronze.alias("source"),
        "target.Order_ID = source.Order_ID AND target.Product_ID = source.Product_ID"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_bronze.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.{table}")

print("Data written to bronze table successfully")

In [0]:
%sql 

-- check row count 
select count(*) as total_rows,count(distinct Order_ID,Product_ID) as distinct_business_keys  from global_mart_retail_dev.bronze.superstore_orders 

In [0]:

%sql 
with cte_dups as (
select  Order_ID,Product_ID , count(*) as dups  from global_mart_retail_dev.bronze.superstore_orders group by Order_ID,Product_ID having count(*) >1
)

select * from global_mart_retail_dev.bronze.superstore_orders as o
inner join cte_dups as d on o.Order_ID = d.Order_ID and o.Product_ID = d.Product_ID



In [0]:
%sql
SELECT * 
FROM global_mart_retail_dev.bronze.superstore_orders
WHERE Customer_Name LIKE '%�%' 
   OR Product_Name LIKE '%�%'